In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

builder = (SparkSession.builder
           .appName("checkpoints")
           .master("spark://spark-master:7077")
           .config("spark.executor.memory", "512m")
           .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
           .config("spark.sql.catalog.spark-catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder, ['org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.1']).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-core_2.12 added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-57ef1548-82e2-4f8d-aa7d-a35dc457bba2;1.0
	confs: [default]
	found io.delta#delta-core_2.12;2.4.0 in central
	found io.delta#delta-storage;2.4.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.1 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in centra

In [2]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka:9092")
      .option("subscribe", "events")
      .option("startingOffsets", "earliest")
      .load())

In [3]:
schema = StructType([
    StructField('user_id', IntegerType(), True),
    StructField('event_type', StringType(), True),
    StructField('event_time', StringType(), True),
    StructField('processing_time', StringType(), True)])

df = df.withColumn('value', from_json(col('value').cast("STRING"), schema))

In [4]:
df = (df
      .select(
          col('value.user_id').alias('user_id'),
          col('value.event_type').alias('event_type'),
          col('value.event_time').alias('event_time'),
          col('value.processing_time').alias('processing_time'))
      .withColumn("event_time"
        , to_timestamp(col("event_time")
        , "MM/dd/yyyy, HH:mm:ss" ))
      .withColumn("processing_time"
        , to_timestamp(col("processing_time")
        , "MM/dd/yyyy, HH:mm:ss")))

In [5]:
df = (df
      .groupBy(
          window(col("event_time"), "60 minute", "60 minute").alias("TimeWindow"),
          col("event_type").alias("EventType"))
      .agg(count(col("user_id")).alias("NumberOfUsers")))

In [6]:
query = (df.writeStream
         .outputMode('complete')
         .format('console')
         .option("truncate", False)
         .start())

-------------------------------------------
Batch: 0
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |2            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |2            |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 1
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |2            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |2            |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 2
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |2            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |3            |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |2            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |3            |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 4
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |2            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |1            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |2            |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |3            |
+------------------------------------------+---------+-------------+



...

-------------------------------------------
Batch: 65
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |15           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |10           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |14           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |16           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |15           |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 66
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |15           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |11           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |14           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |16           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |15           |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 67
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |15           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |11           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |14           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |16           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |16           |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 68
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |15           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |12           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |14           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |16           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |16           |
+------------------------------------------+---------+-------------+



-------------------------------------------
Batch: 69
-------------------------------------------
+------------------------------------------+---------+-------------+
|TimeWindow                                |EventType|NumberOfUsers|
+------------------------------------------+---------+-------------+
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|share    |15           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|view     |13           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|like     |14           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|click    |16           |
|{2025-06-03 10:00:00, 2025-06-03 11:00:00}|purchase |16           |
+------------------------------------------+---------+-------------+



In [7]:
query.stop()

25/06/03 10:29:32 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 70, writer: ConsoleWriter[numRows=20, truncate=false]] is aborting.
25/06/03 10:29:32 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 70, writer: ConsoleWriter[numRows=20, truncate=false]] aborted.


In [8]:
spark.stop()